# Lumbar Spine MRI Segmentation Baseline
## Modified U-Net with Combined Loss (Focal + Dice)

Reproduction baseline based on *Pioneering Precision in Lumbar Spine MRI Segmentation with Advanced Deep Learning and Data Enhancement* (Ahmed et al., 2025).

This notebook implements the paper-aligned pipeline:

1. Read SPIDER MHA volumes from Google Drive.
2. Convert SPIDER labels to 4 classes.
3. Extract sagittal 2D slices at 512x640.
4. Filter incomplete / highly imbalanced slices.
5. Train a Keras Modified U-Net with Leaky ReLU, Glorot initialization, and Combined Loss.
6. Evaluate Dice, IoU, Precision, Recall, and F1.

---


## 0. Environment Setup

In [ ]:
# Google Drive mount
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install SimpleITK for reading MHA files
!pip install SimpleITK -q

In [ ]:
import os
import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

# Paths
BASE_DIR = Path('/content/drive/MyDrive/SPIDER/DataSet')
IMAGE_DIR = BASE_DIR / 'images'
MASK_DIR = BASE_DIR / 'masks'

print(f'Image dir exists: {IMAGE_DIR.exists()}')
print(f'Mask dir exists: {MASK_DIR.exists()}')

## 1. Data Exploration

SPIDERデータセットの構造を確認する。

### 1.1 File Listing

In [ ]:
# List all files
image_files = sorted(os.listdir(IMAGE_DIR))
mask_files = sorted(os.listdir(MASK_DIR))

print(f'Number of image files: {len(image_files)}')
print(f'Number of mask files: {len(mask_files)}')
print(f'\nFirst 10 image files:')
for f in image_files[:10]:
    print(f'  {f}')
print(f'\nFirst 10 mask files:')
for f in mask_files[:10]:
    print(f'  {f}')

In [ ]:
# Check file extensions
image_extensions = Counter(Path(f).suffix for f in image_files)
mask_extensions = Counter(Path(f).suffix for f in mask_files)
print(f'Image extensions: {dict(image_extensions)}')
print(f'Mask extensions: {dict(mask_extensions)}')

In [ ]:
# Check file naming pattern (T1, T2, T2_SPACE)
def classify_sequence(filename):
    name = filename.lower()
    if 't2_space' in name or 't2_sag_space' in name or 'space' in name:
        return 'T2_SPACE'
    elif 't2' in name:
        return 'T2'
    elif 't1' in name:
        return 'T1'
    else:
        return 'Unknown'

sequence_counts = Counter(classify_sequence(f) for f in image_files)
print(f'Sequence distribution: {dict(sequence_counts)}')
print(f'\nSample filenames by sequence:')
for seq in ['T1', 'T2', 'T2_SPACE', 'Unknown']:
    samples = [f for f in image_files if classify_sequence(f) == seq][:3]
    if samples:
        print(f'  {seq}: {samples}')

### 1.2 MHA File Structure

In [ ]:
# Load first image and mask to inspect
# Filter to only mha files
mha_images = [f for f in image_files if f.endswith('.mha')]
mha_masks = [f for f in mask_files if f.endswith('.mha')]

if not mha_images:
    print('No .mha files found. Listing all files for inspection:')
    for f in image_files[:20]:
        print(f'  {f}')
else:
    sample_img_path = IMAGE_DIR / mha_images[0]
    sample_mask_path = MASK_DIR / mha_masks[0]

    img = sitk.ReadImage(str(sample_img_path))
    mask = sitk.ReadImage(str(sample_mask_path))

    print(f'=== Sample: {mha_images[0]} ===')
    print(f'Image size: {img.GetSize()}')
    print(f'Image spacing: {img.GetSpacing()}')
    print(f'Image origin: {img.GetOrigin()}')
    print(f'Image direction: {img.GetDirection()}')
    print(f'Image pixel type: {img.GetPixelIDTypeAsString()}')
    print(f'\nMask size: {mask.GetSize()}')
    print(f'Mask spacing: {mask.GetSpacing()}')
    print(f'Mask pixel type: {mask.GetPixelIDTypeAsString()}')

    # Convert to numpy
    img_arr = sitk.GetArrayFromImage(img)
    mask_arr = sitk.GetArrayFromImage(mask)
    print(f'\nImage array shape: {img_arr.shape}  (slices, height, width)')
    print(f'Image dtype: {img_arr.dtype}')
    print(f'Image value range: [{img_arr.min()}, {img_arr.max()}]')
    print(f'\nMask array shape: {mask_arr.shape}')
    print(f'Mask dtype: {mask_arr.dtype}')
    print(f'Mask unique values: {np.unique(mask_arr)}')
    print(f'Mask unique count: {len(np.unique(mask_arr))}')

### 1.3 Label Value Investigation

全マスクのラベル値を確認し、16クラス→4クラスのマッピングを決定する。

In [ ]:
# Check label values across multiple masks
all_label_values = set()
label_info = []

for i, fname in enumerate(mha_masks[:20]):  # Check first 20 masks
    mask = sitk.ReadImage(str(MASK_DIR / fname))
    mask_arr = sitk.GetArrayFromImage(mask)
    unique_vals = np.unique(mask_arr)
    all_label_values.update(unique_vals.tolist())
    label_info.append({
        'file': fname,
        'shape': mask_arr.shape,
        'unique_values': unique_vals.tolist(),
        'n_labels': len(unique_vals)
    })
    print(f'{fname}: shape={mask_arr.shape}, labels={unique_vals}, count={len(unique_vals)}')

print(f'\n=== All unique label values across 20 masks ===')
print(sorted(all_label_values))
print(f'Total unique labels: {len(all_label_values)}')

In [ ]:
# Detailed label frequency for ONE mask
if mha_masks:
    mask = sitk.ReadImage(str(MASK_DIR / mha_masks[0]))
    mask_arr = sitk.GetArrayFromImage(mask)
    unique, counts = np.unique(mask_arr, return_counts=True)
    total = mask_arr.size

    print(f'=== Label distribution: {mha_masks[0]} ===')
    print(f'{"Label":>8} {"Count":>12} {"Percentage":>12}')
    print('-' * 35)
    for val, cnt in zip(unique, counts):
        print(f'{val:>8} {cnt:>12} {cnt/total*100:>11.2f}%')

### 1.4 Visualize Sample Slices

In [ ]:
# Visualize middle sagittal slices of first few samples.
def preview_sagittal_axis(arr_shape):
    return int(np.argmin(arr_shape))

if mha_images:
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))

    for row in range(3):
        if row >= len(mha_images):
            break
        img = sitk.ReadImage(str(IMAGE_DIR / mha_images[row]))
        mask = sitk.ReadImage(str(MASK_DIR / mha_masks[row]))
        img_arr = sitk.GetArrayFromImage(img)
        mask_arr = sitk.GetArrayFromImage(mask)

        axis = preview_sagittal_axis(img_arr.shape)
        img_slices = np.moveaxis(img_arr, axis, 0)
        mask_slices = np.moveaxis(mask_arr, axis, 0)
        mid_slice = img_slices.shape[0] // 2
        mapped_mask = map_labels(mask_slices[mid_slice]) if 'map_labels' in globals() else mask_slices[mid_slice]

        axes[row, 0].imshow(img_slices[mid_slice], cmap='gray')
        axes[row, 0].set_title(f'{mha_images[row]}\nMRI axis={axis}')
        axes[row, 0].axis('off')

        axes[row, 1].imshow(mask_slices[mid_slice], cmap='tab20')
        axes[row, 1].set_title('Original mask')
        axes[row, 1].axis('off')

        axes[row, 2].imshow(mapped_mask, cmap='tab10', vmin=0, vmax=3)
        axes[row, 2].set_title('4-class mask')
        axes[row, 2].axis('off')

        axes[row, 3].imshow(img_slices[mid_slice], cmap='gray')
        axes[row, 3].imshow(mapped_mask, cmap='tab10', alpha=0.4, vmin=0, vmax=3)
        axes[row, 3].set_title('Overlay')
        axes[row, 3].axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
# Show multiple sagittal slices of one volume to understand the stack.
if mha_images:
    sample_idx = 0
    img = sitk.ReadImage(str(IMAGE_DIR / mha_images[sample_idx]))
    mask = sitk.ReadImage(str(MASK_DIR / mha_masks[sample_idx]))
    img_arr = sitk.GetArrayFromImage(img)
    mask_arr = sitk.GetArrayFromImage(mask)

    axis = preview_sagittal_axis(img_arr.shape)
    img_slices = np.moveaxis(img_arr, axis, 0)
    mask_slices = np.moveaxis(mask_arr, axis, 0)

    n_show = min(8, img_slices.shape[0])
    slice_indices = np.linspace(0, img_slices.shape[0] - 1, n_show, dtype=int)

    fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 6))
    for col, s in enumerate(slice_indices):
        axes[0, col].imshow(img_slices[s], cmap='gray')
        axes[0, col].set_title(f'Slice {s}')
        axes[0, col].axis('off')

        axes[1, col].imshow(mask_slices[s], cmap='tab20')
        axes[1, col].axis('off')

    plt.suptitle(f'{mha_images[sample_idx]} sagittal axis={axis}')
    plt.tight_layout()
    plt.show()


### 1.5 Summary Statistics

In [ ]:
# Collect shape/spacing info - header only (fast)
print('Collecting metadata (header only, no pixel data)...\n')

metadata = []
for fname in mha_images:
    reader = sitk.ImageFileReader()
    reader.SetFileName(str(IMAGE_DIR / fname))
    reader.ReadImageInformation()
    metadata.append({
        'file': fname,
        'sequence': classify_sequence(fname),
        'size': reader.GetSize(),
        'spacing': tuple(round(s, 3) for s in reader.GetSpacing()),
    })

# Summary by sequence type
for seq in ['T1', 'T2', 'T2_SPACE', 'Unknown']:
    seq_data = [m for m in metadata if m['sequence'] == seq]
    if not seq_data:
        continue
    sizes = [m['size'] for m in seq_data]
    spacings = [m['spacing'] for m in seq_data]
    print(f'=== {seq} ({len(seq_data)} files) ===')
    print(f'  Size range: {min(sizes)} - {max(sizes)}')
    print(f'  Spacing range: {min(spacings)} - {max(spacings)}')
    print()

### 1.6 Check for Overview CSV

SPIDERデータセットにはtrain/val分割やメタデータを含むCSVがあるはず。

In [ ]:
# Look for CSV or other metadata files
spider_root = Path('/content/drive/MyDrive/SPIDER')
print('Files in SPIDER root:')
for f in sorted(spider_root.rglob('*')):
    if f.is_file() and not f.name.startswith('.'):
        rel = f.relative_to(spider_root)
        print(f'  {rel}  ({f.stat().st_size / 1024:.1f} KB)')

In [ ]:
# If overview CSV exists, load and display
import glob
csv_files = list(spider_root.rglob('*.csv')) + list(spider_root.rglob('*.json'))
print(f'Found metadata files: {[str(f.relative_to(spider_root)) for f in csv_files]}')

if csv_files:
    import pandas as pd
    for csv_path in csv_files:
        if csv_path.suffix == '.csv':
            df = pd.read_csv(csv_path)
            print(f'\n=== {csv_path.name} ===')
            print(f'Shape: {df.shape}')
            print(f'Columns: {list(df.columns)}')
            print(df.head(10))

---
## 2. Data Preprocessing

3D MHA → 2Dスライス抽出 → 4クラスマッピング → フィルタリング

### 2.1 Configuration

In [ ]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import cv2
import random
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# === Paths ===
BASE_DIR = Path('/content/drive/MyDrive/SPIDER/DataSet')
IMAGE_DIR = BASE_DIR / 'images'
MASK_DIR = BASE_DIR / 'masks'
CSV_PATH = BASE_DIR / 'SPIDER Lumbar Spine Segmentation Overview.csv'

OUTPUT_DIR = Path('/content/drive/MyDrive/SPIDER/processed_baseline')
OUTPUT_IMG_DIR = OUTPUT_DIR / 'images'
OUTPUT_MASK_DIR = OUTPUT_DIR / 'masks'
OUTPUT_IMG_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_MASK_DIR.mkdir(parents=True, exist_ok=True)

# === Reprocessing controls ===
FORCE_REPROCESS = False
RUN_SEQUENCES = None  # None = all sequences; or set to {'T1'}, {'T2'}, {'T2_SPACE'}

# === Hyperparameters from / inferred from paper ===
TARGET_H, TARGET_W = 512, 640
NUM_CLASSES = 4
CLASS_IMBALANCE_THRESHOLD = 0.55
MIN_CLASSES_REQUIRED = 4
FOCAL_GAMMA = 4.0
FOCAL_WEIGHT = 0.6
BATCH_SIZE = 8
EPOCHS = 100
LEAKY_RELU_ALPHA = 0.1
DROPOUT_RATE = 0.5  # Paper does not specify rate; 0.5 is a conservative baseline default.
LEARNING_RATE = 1e-4  # Paper does not specify optimizer/lr; Adam 1e-4 is used as baseline.

CLASS_NAMES = ['Background', 'Vertebrae', 'Spinal Canal', 'IVDs']
CLASS_COLORS = np.array([[0, 0, 0], [255, 0, 0], [0, 255, 0], [0, 0, 255]], dtype=np.uint8)

# === Label Mapping confirmed from SPIDER masks ===
# 0       -> 0 (Background)
# 1-99    -> 1 (Vertebrae)
# 100     -> 2 (Spinal Canal)
# 200+    -> 3 (IVDs)
def map_labels(mask):
    """Map SPIDER labels to the 4 classes used by Ahmed et al. (2025)."""
    new_mask = np.zeros_like(mask, dtype=np.uint8)
    new_mask[(mask >= 1) & (mask <= 99)] = 1
    new_mask[mask == 100] = 2
    new_mask[mask >= 200] = 3
    return new_mask

print('Configuration loaded.')
print(f'Image dir: {IMAGE_DIR}')
print(f'Mask dir: {MASK_DIR}')
print(f'Output directory: {OUTPUT_DIR}')


### 2.2 Extract 2D Slices & Apply Label Mapping

3D MHA から矢状断の2Dスライスを抽出し、18ラベル→4クラスに変換して保存する。

In [ ]:
def infer_sagittal_axis(arr_shape):
    """Infer the sagittal slice axis from a SPIDER volume array shape.

    SimpleITK returns arrays as (z, y, x), but SPIDER files are not uniform:
    some sagittal series have the slice dimension in axis 0, while others have it
    in axis 2. The sagittal stack usually has the smallest dimension, so this
    baseline uses the smallest axis as the slice axis.
    """
    return int(np.argmin(arr_shape))


def iter_sagittal_slices(volume, mask):
    """Yield image/mask slices after moving the inferred sagittal axis to front."""
    axis = infer_sagittal_axis(volume.shape)
    volume_slices = np.moveaxis(volume, axis, 0)
    mask_slices = np.moveaxis(mask, axis, 0)
    for idx in range(volume_slices.shape[0]):
        yield idx, axis, volume_slices[idx], mask_slices[idx]


def should_use_file(fname):
    if RUN_SEQUENCES is None:
        return True
    return classify_sequence(fname) in RUN_SEQUENCES


def extract_and_save_slices(image_dir, mask_dir, output_img_dir, output_mask_dir,
                            target_h, target_w, force=False):
    """Extract sagittal 2D slices, map labels, resize, and save paired .npz files."""
    mha_files = sorted([f for f in os.listdir(image_dir) if f.endswith('.mha') and should_use_file(f)])
    stats = {'total_slices': 0, 'files_processed': 0, 'skipped_existing': 0, 'axes': {}, 'errors': []}

    for fname in tqdm(mha_files, desc='Extracting sagittal slices'):
        mask_path = mask_dir / fname
        if not mask_path.exists():
            stats['errors'].append(f'No mask for {fname}')
            continue

        base_name = fname.replace('.mha', '')
        existing = list(output_img_dir.glob(f'{base_name}_s*.npz'))
        if existing and not force:
            stats['skipped_existing'] += 1
            stats['total_slices'] += len(existing)
            continue

        try:
            img_sitk = sitk.ReadImage(str(image_dir / fname))
            mask_sitk = sitk.ReadImage(str(mask_path))

            img_arr = sitk.GetArrayFromImage(img_sitk).astype(np.float32)
            mask_arr = sitk.GetArrayFromImage(mask_sitk).astype(np.int16)

            if img_arr.shape != mask_arr.shape:
                stats['errors'].append(f'{fname}: image/mask shape mismatch {img_arr.shape} vs {mask_arr.shape}')
                continue

            axis = infer_sagittal_axis(img_arr.shape)
            stats['axes'][axis] = stats['axes'].get(axis, 0) + 1

            for s, _, img_slice, mask_slice in iter_sagittal_slices(img_arr, mask_arr):
                if img_slice.max() == img_slice.min():
                    continue

                mask_4class = map_labels(mask_slice)

                img_resized = cv2.resize(img_slice, (target_w, target_h), interpolation=cv2.INTER_LINEAR)
                mask_resized = cv2.resize(mask_4class, (target_w, target_h), interpolation=cv2.INTER_NEAREST)

                slice_name = f'{base_name}_s{s:03d}'
                np.savez_compressed(output_img_dir / f'{slice_name}.npz', image=img_resized.astype(np.float32))
                np.savez_compressed(output_mask_dir / f'{slice_name}.npz', mask=mask_resized.astype(np.uint8))
                stats['total_slices'] += 1

            stats['files_processed'] += 1

        except Exception as e:
            stats['errors'].append(f'{fname}: {str(e)}')

    return stats


stats = extract_and_save_slices(
    IMAGE_DIR, MASK_DIR, OUTPUT_IMG_DIR, OUTPUT_MASK_DIR,
    TARGET_H, TARGET_W, force=FORCE_REPROCESS
)

print(f"\nFiles processed: {stats['files_processed']}")
print(f"Files skipped because outputs already exist: {stats['skipped_existing']}")
print(f"Total slices available/extracted: {stats['total_slices']}")
print(f"Inferred sagittal axis counts: {stats['axes']}")
if stats['errors']:
    print(f"Errors ({len(stats['errors'])}):")
    for e in stats['errors'][:10]:
        print(f'  {e}')


### 2.3 Data Filtering

論文に従い、以下の基準でスライスをフィルタリング:
1. 4クラス未満のスライスを除外
2. クラス不均衡比率 > 55% のスライスを除外

In [ ]:
def foreground_class_fractions(mask):
    """Return class fractions among foreground pixels only."""
    foreground = mask[mask > 0]
    if foreground.size == 0:
        return {}
    unique, counts = np.unique(foreground, return_counts=True)
    total = counts.sum()
    return {int(u): float(c / total) for u, c in zip(unique, counts)}


def dominant_foreground_fraction(mask):
    fractions = foreground_class_fractions(mask)
    return max(fractions.values()) if fractions else 1.0


def filter_slices(img_dir, mask_dir, min_classes, imbalance_threshold):
    """Filter slices based on paper criteria.

    The paper describes a 55% class-imbalance threshold ambiguously. For this
    baseline, a slice is removed when any foreground class occupies more than
    55% of foreground pixels after requiring all 4 classes to be present.
    """
    mask_files = sorted([f for f in os.listdir(mask_dir) if f.endswith('.npz')])

    kept = []
    removed_class_count = 0
    removed_imbalance = 0
    rows = []

    for fname in tqdm(mask_files, desc='Filtering slices'):
        mask = np.load(mask_dir / fname)['mask']
        unique_classes = np.unique(mask)

        if len(unique_classes) < min_classes:
            removed_class_count += 1
            continue

        max_fraction = dominant_foreground_fraction(mask)
        if max_fraction > imbalance_threshold:
            removed_imbalance += 1
            continue

        fractions = foreground_class_fractions(mask)
        rows.append({
            'file': fname,
            'max_foreground_fraction': max_fraction,
            'vertebrae_fraction': fractions.get(1, 0.0),
            'canal_fraction': fractions.get(2, 0.0),
            'ivd_fraction': fractions.get(3, 0.0),
        })
        kept.append(fname)

    return kept, removed_class_count, removed_imbalance, pd.DataFrame(rows)


kept_files, rm_class, rm_imbalance, filter_df = filter_slices(
    OUTPUT_IMG_DIR, OUTPUT_MASK_DIR, MIN_CLASSES_REQUIRED, CLASS_IMBALANCE_THRESHOLD
)

print(f'\n=== Filtering Results ===')
print(f'Total slices before filtering: {len([f for f in os.listdir(OUTPUT_MASK_DIR) if f.endswith(".npz")])}')
print(f'Removed (< {MIN_CLASSES_REQUIRED} classes): {rm_class}')
print(f'Removed (dominant foreground class > {CLASS_IMBALANCE_THRESHOLD*100:.0f}%): {rm_imbalance}')
print(f'Kept: {len(kept_files)}')

FILTERED_LIST_PATH = OUTPUT_DIR / 'filtered_files.txt'
FILTERED_STATS_PATH = OUTPUT_DIR / 'filtered_slice_stats.csv'
with open(FILTERED_LIST_PATH, 'w') as f:
    for fname in kept_files:
        f.write(fname + '\n')
filter_df.to_csv(FILTERED_STATS_PATH, index=False)

print(f'Filtered file list saved to: {FILTERED_LIST_PATH}')
print(f'Filtered stats saved to: {FILTERED_STATS_PATH}')
if not filter_df.empty:
    display(filter_df.describe())


### 2.4 Train/Val Split & Verify Preprocessed Data

In [ ]:
overview_df = pd.read_csv(CSV_PATH)
train_ids = set(overview_df.loc[overview_df['subset'] == 'training', 'new_file_name'].astype(str).values)
val_ids = set(overview_df.loc[overview_df['subset'] == 'validation', 'new_file_name'].astype(str).values)


def get_series_id(slice_filename):
    """Extract series ID from a slice filename: '100_t1_s005.npz' -> '100_t1'."""
    return slice_filename.replace('.npz', '').rsplit('_s', 1)[0]


train_slices = [f for f in kept_files if get_series_id(f) in train_ids]
val_slices = [f for f in kept_files if get_series_id(f) in val_ids]
unmatched_slices = [f for f in kept_files if get_series_id(f) not in train_ids and get_series_id(f) not in val_ids]

print(f'=== Train/Val Split ===')
print(f'Training slices: {len(train_slices)}')
print(f'Validation slices: {len(val_slices)}')
print(f'Unmatched: {len(unmatched_slices)}')

if len(train_slices) == 0 or len(val_slices) == 0:
    raise ValueError('Train/validation split is empty. Check CSV_PATH and filename matching.')

# Verify a few preprocessed samples.
n_samples = min(3, len(train_slices))
fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4 * n_samples))
if n_samples == 1:
    axes = axes[np.newaxis, :]

for i in range(n_samples):
    idx = int(np.linspace(0, len(train_slices) - 1, n_samples)[i])
    fname = train_slices[idx]

    img = np.load(OUTPUT_IMG_DIR / fname)['image']
    mask = np.load(OUTPUT_MASK_DIR / fname)['mask']

    img_disp = (img - img.min()) / (img.max() - img.min() + 1e-8)
    color_mask = CLASS_COLORS[mask]

    axes[i, 0].imshow(img_disp, cmap='gray')
    axes[i, 0].set_title(fname.replace('.npz', ''))
    axes[i, 0].axis('off')

    axes[i, 1].imshow(color_mask)
    axes[i, 1].set_title(f'Mask classes: {np.unique(mask)}')
    axes[i, 1].axis('off')

    axes[i, 2].imshow(img_disp, cmap='gray')
    axes[i, 2].imshow(color_mask, alpha=0.4)
    axes[i, 2].set_title('Overlay')
    axes[i, 2].axis('off')

legend_text = ' | '.join([f'{idx}: {name}' for idx, name in enumerate(CLASS_NAMES)])
plt.suptitle(f'Preprocessed baseline samples — {legend_text}', fontsize=11)
plt.tight_layout()
plt.show()


---
## 3. Model Definition

Modified U-Net with Leaky ReLU, Glorot Uniform initializer, and custom Combined Loss.

### 3.1 Loss Functions

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

tf.random.set_seed(SEED)

# === Loss Functions ===
# Combined Loss = alpha * Focal + (1 - alpha) * Dice
# Paper baseline: alpha = 0.6, gamma = 4.0

def focal_loss(y_true, y_pred, gamma=FOCAL_GAMMA):
    """Multi-class focal loss for one-hot segmentation masks."""
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
    focal = -y_true * tf.pow(1.0 - y_pred, gamma) * tf.math.log(y_pred)
    return tf.reduce_mean(tf.reduce_sum(focal, axis=-1))


def dice_loss(y_true, y_pred, epsilon=1e-6):
    """Soft Dice loss averaged across batch and classes."""
    numerator = 2.0 * tf.reduce_sum(y_true * y_pred, axis=(1, 2)) + epsilon
    denominator = tf.reduce_sum(y_true + y_pred, axis=(1, 2)) + epsilon
    dice = numerator / denominator
    return 1.0 - tf.reduce_mean(dice)


def combined_loss(alpha=FOCAL_WEIGHT, gamma=FOCAL_GAMMA):
    """Paper-aligned combined focal + dice loss."""
    def loss_fn(y_true, y_pred):
        return alpha * focal_loss(y_true, y_pred, gamma=gamma) + (1.0 - alpha) * dice_loss(y_true, y_pred)
    return loss_fn

print('Loss functions defined.')
print(f'Combined Loss: {FOCAL_WEIGHT} * Focal(gamma={FOCAL_GAMMA}) + {1.0 - FOCAL_WEIGHT:.1f} * Dice')


### 3.2 Modified U-Net Architecture

In [ ]:
def conv_block(x, filters, dropout_rate=DROPOUT_RATE):
    """Conv -> BN -> LeakyReLU -> Conv -> BN -> LeakyReLU -> Dropout"""
    x = layers.Conv2D(filters, 3, padding='same',
                       kernel_initializer='glorot_uniform')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=LEAKY_RELU_ALPHA)(x)
    x = layers.Conv2D(filters, 3, padding='same',
                       kernel_initializer='glorot_uniform')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(alpha=LEAKY_RELU_ALPHA)(x)
    x = layers.Dropout(dropout_rate)(x)
    return x

def upsample_block(x, skip, filters):
    """Conv2DTranspose -> LeakyReLU -> Concatenate with skip connection"""
    x = layers.Conv2DTranspose(filters, 2, strides=2, padding='same',
                                kernel_initializer='glorot_uniform')(x)
    x = layers.LeakyReLU(alpha=LEAKY_RELU_ALPHA)(x)
    x = layers.Concatenate()([x, skip])
    return x

def build_modified_unet(input_shape=(512, 640, 1), num_classes=4, dropout_rate=DROPOUT_RATE):
    """Modified U-Net as described in the paper."""
    inputs = layers.Input(shape=input_shape)

    # --- Encoder (Contractive Path) ---
    # Level 1: 64
    e1 = conv_block(inputs, 64, dropout_rate)
    p1 = layers.MaxPooling2D(2)(e1)

    # Level 2: 128
    e2 = conv_block(p1, 128, dropout_rate)
    p2 = layers.MaxPooling2D(2)(e2)

    # Level 3: 256
    e3 = conv_block(p2, 256, dropout_rate)
    p3 = layers.MaxPooling2D(2)(e3)

    # Level 4: 512
    e4 = conv_block(p3, 512, dropout_rate)
    p4 = layers.MaxPooling2D(2)(e4)

    # --- Bottleneck (extra 512-channel layer from paper) ---
    bottleneck = conv_block(p4, 512, dropout_rate)

    # --- Decoder (Expansive Path) ---
    # Level 4: 512 -> 256
    d4 = upsample_block(bottleneck, e4, 512)
    d4 = conv_block(d4, 256, dropout_rate)

    # Level 3: 256 -> 128
    d3 = upsample_block(d4, e3, 256)
    d3 = conv_block(d3, 128, dropout_rate)

    # Level 2: 128 -> 64
    d2 = upsample_block(d3, e2, 128)
    d2 = conv_block(d2, 64, dropout_rate)

    # Level 1: 64 -> 64
    d1 = upsample_block(d2, e1, 64)
    d1 = conv_block(d1, 64, dropout_rate)

    # --- Output ---
    outputs = layers.Conv2D(num_classes, 1, activation='softmax',
                             kernel_initializer='glorot_uniform')(d1)

    model = Model(inputs, outputs, name='Modified_UNet')
    return model

# Build and show summary
model = build_modified_unet(
    input_shape=(TARGET_H, TARGET_W, 1),
    num_classes=NUM_CLASSES
)
model.summary()
print(f'\nTotal parameters: {model.count_params():,}')


---
## 4. Dataset & Training

### 4.1 TensorFlow Dataset Pipeline

In [ ]:
def load_sample(fname, img_dir, mask_dir, num_classes):
    """Load a single preprocessed image-mask pair from npz files."""
    img = np.load(img_dir / fname)['image'].astype(np.float32)
    mask = np.load(mask_dir / fname)['mask'].astype(np.int32)

    img_min, img_max = img.min(), img.max()
    if img_max > img_min:
        img = (img - img_min) / (img_max - img_min)
    else:
        img = np.zeros_like(img, dtype=np.float32)

    img = img[..., np.newaxis]
    mask_onehot = np.eye(num_classes, dtype=np.float32)[mask]
    return img, mask_onehot


def create_dataset(file_list, img_dir, mask_dir, num_classes, batch_size, shuffle=True):
    """Create a tf.data.Dataset from a list of npz filenames."""
    if len(file_list) == 0:
        raise ValueError('file_list is empty')

    def generator():
        indices = np.arange(len(file_list))
        if shuffle:
            np.random.shuffle(indices)
        for i in indices:
            yield load_sample(file_list[i], img_dir, mask_dir, num_classes)

    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(TARGET_H, TARGET_W, 1), dtype=tf.float32),
            tf.TensorSpec(shape=(TARGET_H, TARGET_W, num_classes), dtype=tf.float32),
        )
    )
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


train_ds = create_dataset(train_slices, OUTPUT_IMG_DIR, OUTPUT_MASK_DIR, NUM_CLASSES, BATCH_SIZE, shuffle=True)
val_ds = create_dataset(val_slices, OUTPUT_IMG_DIR, OUTPUT_MASK_DIR, NUM_CLASSES, BATCH_SIZE, shuffle=False)

for img_batch, mask_batch in train_ds.take(1):
    print(f'Image batch shape: {img_batch.shape}')
    print(f'Mask batch shape:  {mask_batch.shape}')
    print(f'Image range: [{img_batch.numpy().min():.3f}, {img_batch.numpy().max():.3f}]')
    print(f'Mask one-hot sum check: {mask_batch.numpy()[0, 0, 0].sum()}')


### 4.2 Metrics

In [ ]:
def mean_iou(y_true, y_pred):
    """Mean IoU metric for Keras."""
    y_pred_argmax = tf.argmax(y_pred, axis=-1)
    y_true_argmax = tf.argmax(y_true, axis=-1)

    iou_sum = 0.0
    for c in range(NUM_CLASSES):
        pred_c = tf.cast(tf.equal(y_pred_argmax, c), tf.float32)
        true_c = tf.cast(tf.equal(y_true_argmax, c), tf.float32)
        intersection = tf.reduce_sum(pred_c * true_c)
        union = tf.reduce_sum(pred_c) + tf.reduce_sum(true_c) - intersection
        iou_sum += (intersection + 1e-7) / (union + 1e-7)

    return iou_sum / NUM_CLASSES

def dice_coefficient(y_true, y_pred):
    """Dice coefficient metric for Keras."""
    y_pred_argmax = tf.argmax(y_pred, axis=-1)
    y_true_argmax = tf.argmax(y_true, axis=-1)

    dice_sum = 0.0
    for c in range(NUM_CLASSES):
        pred_c = tf.cast(tf.equal(y_pred_argmax, c), tf.float32)
        true_c = tf.cast(tf.equal(y_true_argmax, c), tf.float32)
        intersection = tf.reduce_sum(pred_c * true_c)
        dice_sum += (2.0 * intersection + 1e-7) / (tf.reduce_sum(pred_c) + tf.reduce_sum(true_c) + 1e-7)

    return dice_sum / NUM_CLASSES

print('Metrics defined: mean_iou, dice_coefficient')

### 4.3 Compile & Train

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=combined_loss(alpha=FOCAL_WEIGHT, gamma=FOCAL_GAMMA),
    metrics=['accuracy', mean_iou, dice_coefficient]
)

CHECKPOINT_DIR = Path('/content/drive/MyDrive/SPIDER/checkpoints_baseline')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(CHECKPOINT_DIR / 'best_model.keras'),
        monitor='val_mean_iou',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_mean_iou',
        mode='max',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_mean_iou',
        mode='max',
        factor=0.5,
        patience=7,
        min_lr=1e-7,
        verbose=1
    ),
    keras.callbacks.CSVLogger(str(CHECKPOINT_DIR / 'training_log.csv')),
]

print('Model compiled. Ready to train.')
print(f'Optimizer: Adam(lr={LEARNING_RATE})')
print(f'Checkpoints will be saved to: {CHECKPOINT_DIR}')


In [ ]:
# Train
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

# Save final model
model.save(str(CHECKPOINT_DIR / 'final_model.keras'))
print(f'Training complete. Model saved to {CHECKPOINT_DIR}')

---
## 5. Evaluation & Visualization

### 5.1 Training Curves

In [ ]:
def plot_training_curves(history):
    """Plot accuracy, dice, mean IoU, and loss curves."""
    metrics = {
        'Accuracy': ('accuracy', 'val_accuracy'),
        'Dice Coefficient': ('dice_coefficient', 'val_dice_coefficient'),
        'Mean IoU': ('mean_iou', 'val_mean_iou'),
        'Loss': ('loss', 'val_loss'),
    }

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for ax, (title, (train_key, val_key)) in zip(axes.flat, metrics.items()):
        if train_key in history.history:
            ax.plot(history.history[train_key], label='Train')
            ax.plot(history.history[val_key], label='Validation')
            ax.set_title(title)
            ax.set_xlabel('Epoch')
            ax.set_ylabel(title)
            ax.legend()
            ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(str(CHECKPOINT_DIR / 'training_curves.png'), dpi=150)
    plt.show()
    print(f'Training curves saved to {CHECKPOINT_DIR / "training_curves.png"}')

plot_training_curves(history)

### 5.2 Class-wise Evaluation on Validation Set

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

EVAL_LIMIT = None  # Set to an integer for a quick smoke evaluation.


def evaluate_classwise(model, file_list, img_dir, mask_dir, num_classes):
    """Compute class-wise Dice, IoU, Precision, Recall, and F1."""
    if len(file_list) == 0:
        raise ValueError('No files provided for evaluation')

    all_dice = {c: [] for c in range(num_classes)}
    all_iou = {c: [] for c in range(num_classes)}
    all_preds = []
    all_trues = []

    for fname in tqdm(file_list, desc='Evaluating'):
        img, mask_oh = load_sample(fname, img_dir, mask_dir, num_classes)
        pred = model.predict(img[np.newaxis, ...], verbose=0)[0]

        pred_cls = np.argmax(pred, axis=-1).flatten()
        true_cls = np.argmax(mask_oh, axis=-1).flatten()
        all_preds.append(pred_cls)
        all_trues.append(true_cls)

        for c in range(num_classes):
            pred_c = (pred_cls == c)
            true_c = (true_cls == c)
            intersection = np.logical_and(pred_c, true_c).sum()
            union = np.logical_or(pred_c, true_c).sum()

            dice = (2.0 * intersection + 1e-7) / (pred_c.sum() + true_c.sum() + 1e-7)
            iou = (intersection + 1e-7) / (union + 1e-7)
            all_dice[c].append(dice)
            all_iou[c].append(iou)

    all_preds = np.concatenate(all_preds)
    all_trues = np.concatenate(all_trues)

    rows = []
    for c in range(num_classes):
        rows.append({
            'class': CLASS_NAMES[c],
            'dice': np.mean(all_dice[c]),
            'iou': np.mean(all_iou[c]),
            'precision': precision_score(all_trues == c, all_preds == c, zero_division=0),
            'recall': recall_score(all_trues == c, all_preds == c, zero_division=0),
            'f1': f1_score(all_trues == c, all_preds == c, zero_division=0),
        })

    results_df = pd.DataFrame(rows)
    mean_row = {'class': 'Mean'}
    for col in ['dice', 'iou', 'precision', 'recall', 'f1']:
        mean_row[col] = results_df[col].mean()
    results_df = pd.concat([results_df, pd.DataFrame([mean_row])], ignore_index=True)

    display(results_df)
    results_df.to_csv(CHECKPOINT_DIR / 'validation_metrics.csv', index=False)
    print(f'Results saved to: {CHECKPOINT_DIR / "validation_metrics.csv"}')
    return results_df


eval_files = val_slices if EVAL_LIMIT is None else val_slices[:EVAL_LIMIT]
val_metrics = evaluate_classwise(model, eval_files, OUTPUT_IMG_DIR, OUTPUT_MASK_DIR, NUM_CLASSES)


### 5.3 Prediction Visualization

MRI画像にセグメンテーション結果をオーバーレイして表示。

In [ ]:
def visualize_predictions(model, file_list, img_dir, mask_dir, num_classes, n_samples=6):
    """Visualize predictions: Original | Ground Truth | Prediction | Overlay"""
    CLASS_COLORS = np.array([[0, 0, 0], [255, 0, 0], [0, 255, 0], [0, 0, 255]], dtype=np.uint8)
    CLASS_NAMES = ['BG', 'Vertebrae', 'Spinal Canal', 'IVDs']

    indices = np.linspace(0, len(file_list) - 1, n_samples, dtype=int)
    fig, axes = plt.subplots(n_samples, 4, figsize=(16, 4 * n_samples))

    for row, idx in enumerate(indices):
        fname = file_list[idx]
        img, mask_oh = load_sample(fname, img_dir, mask_dir, num_classes)

        pred = model.predict(img[np.newaxis, ...], verbose=0)[0]
        pred_cls = np.argmax(pred, axis=-1)
        true_cls = np.argmax(mask_oh, axis=-1)

        img_disp = img[..., 0]
        gt_color = CLASS_COLORS[true_cls]
        pred_color = CLASS_COLORS[pred_cls]

        # Original
        axes[row, 0].imshow(img_disp, cmap='gray')
        axes[row, 0].set_title('MRI' if row == 0 else '')
        axes[row, 0].axis('off')

        # Ground Truth
        axes[row, 1].imshow(gt_color)
        axes[row, 1].set_title('Ground Truth' if row == 0 else '')
        axes[row, 1].axis('off')

        # Prediction
        axes[row, 2].imshow(pred_color)
        axes[row, 2].set_title('Prediction' if row == 0 else '')
        axes[row, 2].axis('off')

        # Overlay
        axes[row, 3].imshow(img_disp, cmap='gray')
        axes[row, 3].imshow(pred_color, alpha=0.4)
        axes[row, 3].set_title('Overlay' if row == 0 else '')
        axes[row, 3].axis('off')

        # Add filename
        axes[row, 0].set_ylabel(fname.replace('.npz', ''), fontsize=8)

    # Color legend
    legend = ' | '.join([f'{n}: {c}' for n, c in
                          zip(CLASS_NAMES, ['Black', 'Red', 'Green', 'Blue'])])
    plt.suptitle(f'Segmentation Results — {legend}', fontsize=12)
    plt.tight_layout()
    plt.savefig(str(CHECKPOINT_DIR / 'predictions.png'), dpi=150)
    plt.show()

visualize_predictions(model, val_slices, OUTPUT_IMG_DIR, OUTPUT_MASK_DIR, NUM_CLASSES)